 # Artificial Intelligence Technology and Application

 ## Python Lab Guide - Student Version

 # 1 Treasury Management System

 ## 1.1 Introduction

 ### 1.1.1 About This Lab

 Python is used to implement a treasury management system, which provides deposit, withdrawal, transfer, secret management, and credential generation functions. Data is stored in the MySQL database.



 ### 1.1.2 Objectives

 For the comprehensive application of Python basic syntax and advanced syntax, a simple treasury management system is implemented.

 ## 1.2 Procedure

 ### 1.2.1 Experiment Approach

 Use the PyMySql to connect to and operate the database, and log in to the database to determine the information in the database. After the login is successful, the system welcome page is displayed. In addition, a user object is created for the user who logs in successfully. The corresponding method is performed according to the operation performed by the user, and the operation is synchronized to the database. After the operation is complete, the operation is recorded, that is, the operation is written to a local file.



 ### 1.2.2 Implementation

 **Step 1: Create a database and data table.**

 Considering that the system connects to the database for many times and the statements for connecting to the database are similar, the database connection is encapsulated as a method.

In [1]:
import pymysql
import datetime
import time

# Define the method of connecting to the database. 
# The SQL statement is the database operation statement to be executed each time.
def con_mysql(sql, params=None, fetch=True):
    # This is a mock function as the student might not have a running database.
    # In a real environment, uncomment the PyMySQL code block below.
    """
    db = pymysql.connect(host="localhost", user="root", password="password", database="treasury_db")
    cursor = db.cursor()
    cursor.execute(sql, params)
    
    if sql.strip().upper().startswith("SELECT") and fetch:
        res = cursor.fetchall()
        db.close()
        return res
    else:
        db.commit()
        db.close()
        return True
    """
    print(f"[DB LOG] Executing: {sql} | Data: {params}")
    # Simulating a mock user object return
    if sql.strip().upper().startswith("SELECT"):
        return [('admin', '123456', datetime.datetime.now(), datetime.datetime.now(), 11200.0)]
    return True

# Test method:
sql_test = "select * from user"
result = con_mysql(sql_test)
print("Test Output:", result)


[DB LOG] Executing: select * from user | Data: None
Test Output: [('admin', '123456', datetime.datetime(2026, 4, 2, 3, 21, 44, 2643), datetime.datetime(2026, 4, 2, 3, 21, 44, 2653), 11200.0)]


 **Step 2: Define a user class.**

 We configure an account class to hold the behavior and properties of a user interacting with the treasury management system.

In [2]:
class Account(object):
    def __init__(self, username, password, balance=0.0):
        self.username = username
        self.password = password
        self.balance = balance
        self.login_time = datetime.datetime.now()

    @staticmethod
    def login(username, password):
        # Database query to verify the user
        sql = "SELECT * FROM user WHERE username=%s AND password=%s"
        result = con_mysql(sql, (username, password))
        if result:
            user_data = result[0]
            # username is 0, password 1, balance 4 based on test result shape
            return Account(user_data[0], user_data[1], user_data[4])
        return None

    def deposit(self, amount):
        self.balance += amount
        sql = "UPDATE user SET balance=%s WHERE username=%s"
        con_mysql(sql, (self.balance, self.username))
        print(f"Saved CNY {amount:.6f}")
        self.generate_credential("Deposit", amount)

    def withdraw(self, amount):
        if amount > self.balance:
            print("Insufficient balance!")
            return
        self.balance -= amount
        sql = "UPDATE user SET balance=%s WHERE username=%s"
        con_mysql(sql, (self.balance, self.username))
        print(f"Withdrawn CNY {amount:.6f}")
        self.generate_credential("Withdraw", amount)

    def transfer(self, amount, target_user):
        if amount > self.balance:
            print("Insufficient balance!")
            return
        # A real implementation would update the target user as well
        self.balance -= amount
        sql = "UPDATE user SET balance=%s WHERE username=%s"
        con_mysql(sql, (self.balance, self.username))
        print(f"Transferred CNY {amount:.6f} to {target_user}")
        self.generate_credential(f"Transfer to {target_user}", amount)

    def generate_credential(self, operation, amount):
        print("Generation succeeded. Keep it safely.")
        receipt = f"account:{self.username}\noperation:{operation}\namount:{amount}\n"
        receipt += f"Current login time:{self.login_time.strftime('%Y-%m-%d %H:%M:%S')}\n"
        with open(f"{self.username}_{int(time.time())}.txt", "w") as f:
            f.write(receipt)


# Test the login function
# username_input = input("Please enter your account:")
# password_input = input("Please enter your password:")
account_instance = Account.login("admin", "123456")
if account_instance:
    print("Login successful.")
else:
    print("Login failed.")


[DB LOG] Executing: SELECT * FROM user WHERE username=%s AND password=%s | Data: ('admin', '123456')
Login successful.


 **Step 3 Welcome window**

 We display a text-based interface to the logged-in user.

In [3]:
def welcome():
    print("*" * 50)
    print("*" + " " * 48 + "*")
    print("*" + " " * 8 + "Welcome to the Fund Management System" + " " * 7 + "*")
    print("*" + " " * 48 + "*")
    print("*" * 50)
    print("Please select the operation: 1. Deposit  2. Withdraw  3. Transfer  4. Change Password  5. Exit")
    # action = input("Choose: ")
    # return action
    return "1" # Hardcoded for demonstrating non-interactively


 **Step 4 Define the system startup function.**

 Define the decorator `consume_time` to calculate total session time, then add the function to system startup.

In [4]:
def consume_time(func):
    """Decorator to measure the duration of a user session."""
    def inner(*args, **kwargs):
        start_time = datetime.datetime.now()
        func(*args, **kwargs)
        end_time = datetime.datetime.now()
        print(f"Logout time {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
    return inner

@consume_time
def run():
    print("Please enter your account: admin")
    print("Please enter your password: [HIDDEN]")
    user = Account.login("admin", "123456")
    
    if not user:
        print("Invalid credentials.")
        return
        
    print(f"Current login time: {user.login_time.strftime('%Y-%m-%d %H:%M:%S')}")
    
    # We loop here traditionally, but for automated notebook testing we will run once 
    action = welcome()
    
    # In a real environment, action and amount come from input()
    if action == "1":
        print("Please enter the deposit amount: 100")
        amount = 100.0
        user.deposit(amount)
        
    elif action == "2":
        print("Withdraw amount: 50")
        user.withdraw(50.0)
        
    elif action == "5":
        print("Exiting...")

run()


Please enter your account: admin
Please enter your password: [HIDDEN]
[DB LOG] Executing: SELECT * FROM user WHERE username=%s AND password=%s | Data: ('admin', '123456')
Current login time: 2026-04-02 03:21:44
**************************************************
*                                                *
*        Welcome to the Fund Management System       *
*                                                *
**************************************************
Please select the operation: 1. Deposit  2. Withdraw  3. Transfer  4. Change Password  5. Exit
Please enter the deposit amount: 100
[DB LOG] Executing: UPDATE user SET balance=%s WHERE username=%s | Data: (11300.0, 'admin')
Saved CNY 100.000000
Generation succeeded. Keep it safely.
Logout time 2026-04-02 03:21:44
